# Introducción a LSTM para Series Temporales

**Elaborado por:** David Palacio J.  
**Correo:** davidpalacioj@gmail.com

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dpalacioj/DataAI-Fundamentos-Aplicaciones/blob/main/InteligenciaArtificial/notebooks_teoria/09_3_LSTM.ipynb)

---

## 🧠 ¿Qué son las LSTM?

**LSTM** (Long Short-Term Memory) son un tipo especial de **redes neuronales recurrentes** diseñadas para recordar información durante períodos largos de tiempo.

### 🤔 **¿Por qué son útiles?**

Imagina que quieres predecir el precio de las acciones de mañana. No solo importa el precio de hoy, sino también:
- Los precios de la última semana
- Las tendencias del último mes
- Los patrones estacionales

Las LSTM son excelentes para **"recordar" patrones importantes** y **"olvidar" información irrelevante**.

### 📊 **¿Cómo funcionan (versión simple)?**

Las LSTM tienen tres "puertas" mágicas:

1. **🚪 Puerta de Olvido**: Decide qué información del pasado olvidar
2. **🚪 Puerta de Entrada**: Decide qué nueva información almacenar
3. **🚪 Puerta de Salida**: Decide qué información usar para la predicción

Es como tener un asistente inteligente que sabe qué recordar y qué ignorar para hacer mejores predicciones.

## 🛠️ Instalación y Librerías

Vamos a usar TensorFlow/Keras para construir nuestra LSTM:

In [ ]:
# Instalar librerías necesarias (solo en Colab)
# !pip install tensorflow pandas numpy matplotlib plotly scikit-learn --quiet

# Importar librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# TensorFlow y Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Utilidades
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Configuración
np.random.seed(42)
tf.random.set_seed(42)

print(f"✅ TensorFlow versión: {tf.__version__}")
print("✅ Librerías importadas exitosamente")

## 📈 Creando una Serie Temporal Sintética

Vamos a crear una serie temporal que simula las **ventas diarias de una tienda** con:
- **Tendencia**: Las ventas crecen con el tiempo
- **Estacionalidad**: Ventas más altas los fines de semana
- **Ruido**: Variación aleatoria diaria

In [ ]:
# Crear serie temporal sintética
np.random.seed(42)

# Parámetros
n_dias = 365 * 2  # 2 años de datos
fechas = pd.date_range(start='2022-01-01', periods=n_dias, freq='D')

# Componentes de la serie
tiempo = np.arange(n_dias)

# 1. Tendencia creciente
tendencia = 100 + tiempo * 0.05

# 2. Estacionalidad semanal (más ventas los fines de semana)
estacionalidad_semanal = 10 * np.sin(2 * np.pi * tiempo / 7)

# 3. Estacionalidad mensual
estacionalidad_mensual = 5 * np.sin(2 * np.pi * tiempo / 30)

# 4. Ruido aleatorio
ruido = np.random.normal(0, 5, n_dias)

# Serie temporal completa
ventas = tendencia + estacionalidad_semanal + estacionalidad_mensual + ruido

# Asegurar que las ventas sean positivas
ventas = np.maximum(ventas, 10)

# Crear DataFrame
df = pd.DataFrame({
    'fecha': fechas,
    'ventas': ventas
})

print(f"📊 Serie temporal creada:")
print(f"  📅 Período: {df['fecha'].min().strftime('%Y-%m-%d')} a {df['fecha'].max().strftime('%Y-%m-%d')}")
print(f"  📈 Total de días: {len(df)}")
print(f"  💰 Ventas promedio: ${df['ventas'].mean():.2f}")
print(f"  📉 Ventas mínimas: ${df['ventas'].min():.2f}")
print(f"  📈 Ventas máximas: ${df['ventas'].max():.2f}")

df.head()

## 📊 Visualizando la Serie Temporal

In [ ]:
# Visualizar la serie temporal completa
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df['fecha'],
    y=df['ventas'],
    mode='lines',
    name='Ventas Diarias',
    line=dict(color='blue', width=1)
))

# Agregar línea de tendencia
z = np.polyfit(tiempo, ventas, 1)
p = np.poly1d(z)
fig.add_trace(go.Scatter(
    x=df['fecha'],
    y=p(tiempo),
    mode='lines',
    name='Tendencia',
    line=dict(color='red', width=2, dash='dash')
))

fig.update_layout(
    title='Serie Temporal: Ventas Diarias de la Tienda',
    xaxis_title='Fecha',
    yaxis_title='Ventas ($)',
    height=400,
    hovermode='x unified'
)
fig.show()

# Visualizar últimos 90 días para ver patrones
df_ultimos = df.tail(90)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=df_ultimos['fecha'],
    y=df_ultimos['ventas'],
    mode='lines+markers',
    name='Ventas',
    line=dict(color='green', width=2),
    marker=dict(size=5)
))

fig2.update_layout(
    title='Últimos 90 Días - Patrones Semanales Visibles',
    xaxis_title='Fecha',
    yaxis_title='Ventas ($)',
    height=400
)
fig2.show()

print("📊 Observaciones:")
print("  📈 Se observa una tendencia creciente en las ventas")
print("  🔄 Hay patrones repetitivos semanales y mensuales")
print("  📊 El ruido aleatorio hace que la predicción sea un desafío interesante")

## 🔧 Preparación de Datos para LSTM

Las LSTM necesitan los datos en un formato especial:
- **Ventanas de tiempo**: Usamos los últimos N días para predecir el siguiente
- **Normalización**: Los valores deben estar entre 0 y 1 para mejor entrenamiento

In [ ]:
# Normalizar los datos (LSTM funciona mejor con valores entre 0 y 1)
scaler = MinMaxScaler(feature_range=(0, 1))
ventas_normalizadas = scaler.fit_transform(df['ventas'].values.reshape(-1, 1))

print("📊 Normalización completada:")
print(f"  Valor mínimo original: ${df['ventas'].min():.2f}")
print(f"  Valor máximo original: ${df['ventas'].max():.2f}")
print(f"  Valor mínimo normalizado: {ventas_normalizadas.min():.4f}")
print(f"  Valor máximo normalizado: {ventas_normalizadas.max():.4f}")

In [ ]:
def crear_ventanas_tiempo(data, ventana_entrada, ventana_salida=1):
    """
    Crea ventanas de tiempo para entrenar la LSTM.
    
    Por ejemplo, si ventana_entrada=30:
    - Usamos los últimos 30 días para predecir el día 31
    """
    X, y = [], []
    
    for i in range(ventana_entrada, len(data) - ventana_salida + 1):
        # Ventana de entrada: últimos N días
        X.append(data[i - ventana_entrada:i, 0])
        # Valor a predecir: siguiente día
        y.append(data[i:i + ventana_salida, 0])
    
    return np.array(X), np.array(y)

# Configuración
VENTANA_ENTRADA = 30  # Usar últimos 30 días
VENTANA_SALIDA = 1    # Predecir 1 día

# Crear ventanas
X, y = crear_ventanas_tiempo(ventas_normalizadas, VENTANA_ENTRADA, VENTANA_SALIDA)

print(f"📊 Ventanas de tiempo creadas:")
print(f"  🔄 Ventana de entrada: {VENTANA_ENTRADA} días")
print(f"  🎯 Ventana de salida: {VENTANA_SALIDA} día")
print(f"  📈 Total de muestras: {len(X)}")
print(f"  📐 Forma de X: {X.shape}")
print(f"  📐 Forma de y: {y.shape}")

In [ ]:
# Dividir en entrenamiento y prueba
split_index = int(len(X) * 0.8)  # 80% entrenamiento, 20% prueba

X_train = X[:split_index]
X_test = X[split_index:]
y_train = y[:split_index]
y_test = y[split_index:]

# Reshape para LSTM [muestras, pasos_tiempo, características]
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

print("📊 División de datos:")
print(f"  🏋️ Entrenamiento: {X_train.shape[0]} muestras")
print(f"  🧪 Prueba: {X_test.shape[0]} muestras")
print(f"  📐 Forma X_train: {X_train.shape}")
print(f"  📐 Forma X_test: {X_test.shape}")

## 🤖 Construyendo el Modelo LSTM

Vamos a crear un modelo LSTM simple pero efectivo:

In [ ]:
# Crear el modelo LSTM
modelo = Sequential([
    # Primera capa LSTM con 50 neuronas
    LSTM(50, activation='relu', return_sequences=True, input_shape=(VENTANA_ENTRADA, 1)),
    Dropout(0.2),  # Prevenir overfitting
    
    # Segunda capa LSTM
    LSTM(50, activation='relu'),
    Dropout(0.2),
    
    # Capa densa para la salida
    Dense(25, activation='relu'),
    Dense(1)  # Una salida: predicción del siguiente día
])

# Compilar el modelo
modelo.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mean_squared_error',
    metrics=['mae']
)

# Resumen del modelo
print("🤖 Arquitectura del Modelo LSTM:")
print("="*50)
modelo.summary()

### 💡 **¿Qué hace cada capa?**

1. **LSTM(50)**: 50 "neuronas con memoria" que aprenden patrones temporales
2. **Dropout(0.2)**: Apaga aleatoriamente 20% de neuronas para evitar memorización
3. **Dense(25)**: Capa tradicional que combina la información
4. **Dense(1)**: Produce la predicción final

## 🏋️ Entrenamiento del Modelo

In [ ]:
# Entrenar el modelo
print("🚀 Iniciando entrenamiento...")
print("(Esto puede tomar 1-2 minutos)\n")

historia = modelo.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,  # Usar 10% para validación
    verbose=0  # Sin output detallado
)

print("✅ Entrenamiento completado!")

# Visualizar el progreso del entrenamiento
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Pérdida (Loss)', 'Error Absoluto Medio (MAE)']
)

# Loss
fig.add_trace(
    go.Scatter(y=historia.history['loss'], name='Entrenamiento', line=dict(color='blue')),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(y=historia.history['val_loss'], name='Validación', line=dict(color='red')),
    row=1, col=1
)

# MAE
fig.add_trace(
    go.Scatter(y=historia.history['mae'], name='Entrenamiento', line=dict(color='blue')),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(y=historia.history['val_mae'], name='Validación', line=dict(color='red')),
    row=1, col=2
)

fig.update_xaxes(title_text="Época")
fig.update_yaxes(title_text="Valor")

fig.update_layout(
    title_text="Progreso del Entrenamiento",
    height=400,
    showlegend=True
)
fig.show()

print("\n📊 Métricas finales:")
print(f"  📉 Loss final: {historia.history['loss'][-1]:.6f}")
print(f"  📏 MAE final: {historia.history['mae'][-1]:.6f}")

## 🔮 Haciendo Predicciones

In [ ]:
# Hacer predicciones
predicciones_train = modelo.predict(X_train, verbose=0)
predicciones_test = modelo.predict(X_test, verbose=0)

# Invertir la normalización
predicciones_train = scaler.inverse_transform(predicciones_train)
predicciones_test = scaler.inverse_transform(predicciones_test)
y_train_real = scaler.inverse_transform(y_train.reshape(-1, 1))
y_test_real = scaler.inverse_transform(y_test.reshape(-1, 1))

print("🔮 Predicciones realizadas")
print(f"  📊 Predicciones de entrenamiento: {len(predicciones_train)}")
print(f"  📊 Predicciones de prueba: {len(predicciones_test)}")

In [ ]:
# Calcular métricas de error
mse_train = mean_squared_error(y_train_real, predicciones_train)
mae_train = mean_absolute_error(y_train_real, predicciones_train)
rmse_train = np.sqrt(mse_train)

mse_test = mean_squared_error(y_test_real, predicciones_test)
mae_test = mean_absolute_error(y_test_real, predicciones_test)
rmse_test = np.sqrt(mse_test)

print("📊 Métricas de Evaluación:")
print("\n🏋️ Conjunto de Entrenamiento:")
print(f"  📏 MAE: ${mae_train:.2f}")
print(f"  📐 RMSE: ${rmse_train:.2f}")

print("\n🧪 Conjunto de Prueba:")
print(f"  📏 MAE: ${mae_test:.2f}")
print(f"  📐 RMSE: ${rmse_test:.2f}")

print(f"\n💡 Interpretación:")
print(f"  En promedio, nuestras predicciones se desvían ${mae_test:.2f} del valor real")

## 📈 Visualización de Resultados

In [ ]:
# Preparar datos para visualización
fechas_train = df['fecha'][VENTANA_ENTRADA:split_index+VENTANA_ENTRADA]
fechas_test = df['fecha'][split_index+VENTANA_ENTRADA:split_index+VENTANA_ENTRADA+len(predicciones_test)]

# Visualizar predicciones vs valores reales
fig = go.Figure()

# Datos de entrenamiento
fig.add_trace(go.Scatter(
    x=fechas_train,
    y=y_train_real.flatten(),
    mode='lines',
    name='Real (Entrenamiento)',
    line=dict(color='blue', width=1),
    opacity=0.6
))

fig.add_trace(go.Scatter(
    x=fechas_train,
    y=predicciones_train.flatten(),
    mode='lines',
    name='Predicción (Entrenamiento)',
    line=dict(color='lightblue', width=1),
    opacity=0.6
))

# Datos de prueba
fig.add_trace(go.Scatter(
    x=fechas_test,
    y=y_test_real.flatten(),
    mode='lines',
    name='Real (Prueba)',
    line=dict(color='green', width=2)
))

fig.add_trace(go.Scatter(
    x=fechas_test,
    y=predicciones_test.flatten(),
    mode='lines',
    name='Predicción (Prueba)',
    line=dict(color='red', width=2, dash='dash')
))

# Línea vertical para separar entrenamiento/prueba
fig.add_vline(
    x=fechas_test.iloc[0],
    line_width=2,
    line_dash="dash",
    line_color="gray",
    annotation_text="Inicio Prueba"
)

fig.update_layout(
    title='Predicciones LSTM vs Valores Reales',
    xaxis_title='Fecha',
    yaxis_title='Ventas ($)',
    height=500,
    hovermode='x unified'
)
fig.show()

print("📊 Observaciones:")
print("  ✅ El modelo captura bien la tendencia general")
print("  ✅ Las predicciones siguen los patrones estacionales")
print("  📈 El modelo generaliza bien en datos no vistos (prueba)")

In [ ]:
# Zoom en los últimos 60 días de prueba
ultimos_60 = 60

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=fechas_test.iloc[-ultimos_60:],
    y=y_test_real[-ultimos_60:].flatten(),
    mode='lines+markers',
    name='Valores Reales',
    line=dict(color='green', width=2),
    marker=dict(size=6)
))

fig2.add_trace(go.Scatter(
    x=fechas_test.iloc[-ultimos_60:],
    y=predicciones_test[-ultimos_60:].flatten(),
    mode='lines+markers',
    name='Predicciones LSTM',
    line=dict(color='red', width=2, dash='dash'),
    marker=dict(size=6, symbol='diamond')
))

fig2.update_layout(
    title='Zoom: Últimos 60 Días - Predicciones Detalladas',
    xaxis_title='Fecha',
    yaxis_title='Ventas ($)',
    height=400,
    hovermode='x unified'
)
fig2.show()

# Calcular error para estos días
error_ultimos = np.abs(y_test_real[-ultimos_60:].flatten() - predicciones_test[-ultimos_60:].flatten())
print(f"\n📊 Análisis de los últimos {ultimos_60} días:")
print(f"  📏 Error promedio: ${error_ultimos.mean():.2f}")
print(f"  📈 Error máximo: ${error_ultimos.max():.2f}")
print(f"  📉 Error mínimo: ${error_ultimos.min():.2f}")

## 🔮 Predicción Futura

Vamos a usar nuestro modelo para predecir los próximos 30 días:

In [ ]:
def predecir_futuro(modelo, ultimo_periodo, n_predicciones, scaler):
    """
    Predice múltiples pasos hacia el futuro.
    """
    predicciones = []
    entrada_actual = ultimo_periodo.copy()
    
    for _ in range(n_predicciones):
        # Hacer predicción
        entrada_reshape = entrada_actual.reshape((1, VENTANA_ENTRADA, 1))
        prediccion = modelo.predict(entrada_reshape, verbose=0)
        predicciones.append(prediccion[0, 0])
        
        # Actualizar ventana: quitar el primer valor y agregar la predicción
        entrada_actual = np.append(entrada_actual[1:], prediccion)
    
    # Desnormalizar predicciones
    predicciones = np.array(predicciones).reshape(-1, 1)
    predicciones = scaler.inverse_transform(predicciones)
    
    return predicciones

# Tomar los últimos 30 días normalizados
ultimo_periodo = ventas_normalizadas[-VENTANA_ENTRADA:].flatten()

# Predecir próximos 30 días
n_dias_futuro = 30
predicciones_futuro = predecir_futuro(modelo, ultimo_periodo, n_dias_futuro, scaler)

# Crear fechas futuras
ultima_fecha = df['fecha'].iloc[-1]
fechas_futuro = pd.date_range(start=ultima_fecha + pd.Timedelta(days=1), periods=n_dias_futuro, freq='D')

print(f"🔮 Predicciones para los próximos {n_dias_futuro} días:")
print(f"  📅 Desde: {fechas_futuro[0].strftime('%Y-%m-%d')}")
print(f"  📅 Hasta: {fechas_futuro[-1].strftime('%Y-%m-%d')}")
print(f"  💰 Ventas promedio predichas: ${predicciones_futuro.mean():.2f}")

In [ ]:
# Visualizar predicciones futuras
fig3 = go.Figure()

# Últimos 90 días históricos
ultimos_historicos = 90
fig3.add_trace(go.Scatter(
    x=df['fecha'].iloc[-ultimos_historicos:],
    y=df['ventas'].iloc[-ultimos_historicos:],
    mode='lines',
    name='Datos Históricos',
    line=dict(color='blue', width=2)
))

# Predicciones futuras
fig3.add_trace(go.Scatter(
    x=fechas_futuro,
    y=predicciones_futuro.flatten(),
    mode='lines+markers',
    name='Predicciones Futuras',
    line=dict(color='red', width=2, dash='dash'),
    marker=dict(size=6)
))

# Área de incertidumbre (simulada)
incertidumbre = predicciones_futuro.flatten() * 0.1  # 10% de incertidumbre
fig3.add_trace(go.Scatter(
    x=fechas_futuro,
    y=predicciones_futuro.flatten() + incertidumbre,
    mode='lines',
    line=dict(width=0),
    showlegend=False,
    hoverinfo='skip'
))

fig3.add_trace(go.Scatter(
    x=fechas_futuro,
    y=predicciones_futuro.flatten() - incertidumbre,
    mode='lines',
    line=dict(width=0),
    fill='tonexty',
    fillcolor='rgba(255,0,0,0.2)',
    name='Banda de Incertidumbre',
    hoverinfo='skip'
))

# Línea vertical para separar histórico/futuro
fig3.add_vline(
    x=df['fecha'].iloc[-1],
    line_width=2,
    line_dash="dash",
    line_color="gray",
    annotation_text="Hoy"
)

fig3.update_layout(
    title='Predicción de Ventas: Próximos 30 Días',
    xaxis_title='Fecha',
    yaxis_title='Ventas ($)',
    height=500,
    hovermode='x unified'
)
fig3.show()

print("\n📊 Resumen de predicciones futuras:")
for i in [0, 6, 13, 20, 29]:
    print(f"  📅 {fechas_futuro[i].strftime('%Y-%m-%d')}: ${predicciones_futuro[i][0]:.2f}")

## 🎯 Análisis de Errores

In [ ]:
# Calcular errores
errores = y_test_real.flatten() - predicciones_test.flatten()

# Visualización de errores
fig4 = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Distribución de Errores', 'Errores vs Tiempo',
                   'Q-Q Plot', 'Errores vs Predicción']
)

# Histograma de errores
fig4.add_trace(
    go.Histogram(x=errores, nbinsx=30, name='Errores'),
    row=1, col=1
)

# Errores a lo largo del tiempo
fig4.add_trace(
    go.Scatter(x=fechas_test, y=errores, mode='lines', name='Error'),
    row=1, col=2
)
fig4.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=2)

# Q-Q plot (aproximado)
errores_sorted = np.sort(errores)
normal_teorico = np.random.normal(0, errores.std(), len(errores))
normal_teorico.sort()

fig4.add_trace(
    go.Scatter(x=normal_teorico, y=errores_sorted, mode='markers', name='Q-Q'),
    row=2, col=1
)

# Errores vs predicción
fig4.add_trace(
    go.Scatter(x=predicciones_test.flatten(), y=errores, mode='markers', name='Error vs Pred'),
    row=2, col=2
)
fig4.add_hline(y=0, line_dash="dash", line_color="red", row=2, col=2)

fig4.update_layout(
    title_text="Análisis de Errores del Modelo LSTM",
    height=600,
    showlegend=False
)
fig4.show()

print("📊 Estadísticas de errores:")
print(f"  📏 Error medio: ${errores.mean():.2f}")
print(f"  📐 Desviación estándar: ${errores.std():.2f}")
print(f"  📈 Error máximo: ${errores.max():.2f}")
print(f"  📉 Error mínimo: ${errores.min():.2f}")
print(f"\n💡 Los errores parecen distribuirse normalmente alrededor de 0, ¡buen signo!")

## 🎓 Resumen y Conceptos Clave

### 📚 **Lo que Aprendimos:**

1. **🧠 LSTM**: Redes neuronales que "recuerdan" patrones en series temporales
2. **📊 Preparación de Datos**: Crear ventanas de tiempo y normalizar
3. **🏗️ Arquitectura**: Capas LSTM + Dropout + Dense
4. **🔮 Predicción**: Usar el pasado reciente para predecir el futuro
5. **📈 Evaluación**: MAE y RMSE para medir el error

### 💡 **Ventajas de LSTM:**

✅ **Captura dependencias largas**: Puede recordar patrones de hace semanas o meses
✅ **Maneja no linealidades**: Aprende relaciones complejas
✅ **Adaptable**: Funciona con diferentes tipos de series temporales
✅ **Robusto al ruido**: Filtra variaciones aleatorias

### ⚠️ **Consideraciones:**

❌ **Requiere muchos datos**: Necesita suficientes ejemplos para aprender
❌ **Tiempo de entrenamiento**: Más lento que métodos tradicionales
❌ **Caja negra**: Difícil de interpretar qué aprende exactamente
❌ **Sensible a hiperparámetros**: Requiere ajuste fino

### 🚀 **Aplicaciones Reales:**

1. **📈 Mercados Financieros**: Predicción de precios de acciones
2. **⚡ Energía**: Predicción de demanda eléctrica
3. **🌤️ Meteorología**: Pronóstico del tiempo
4. **🛒 Retail**: Predicción de ventas y demanda
5. **🏥 Salud**: Predicción de brotes epidémicos

### 🔧 **Cómo Mejorar el Modelo:**

1. **📊 Más datos**: Entrenar con series más largas
2. **🏗️ Arquitectura más profunda**: Más capas LSTM
3. **🎯 Ajuste de hiperparámetros**: Learning rate, batch size, epochs
4. **📈 Features adicionales**: Incluir variables externas (días festivos, promociones)
5. **🔄 Bidirectional LSTM**: Procesar la secuencia en ambas direcciones

### 📖 **Recursos para Profundizar:**

- [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/)
- [TensorFlow Time Series Tutorial](https://www.tensorflow.org/tutorials/structured_data/time_series)
- [Keras LSTM Documentation](https://keras.io/api/layers/recurrent_layers/lstm/)

---

**¡Felicitaciones! Ahora sabes cómo usar LSTM para predecir series temporales 🎯**

Las LSTM son una herramienta poderosa en tu arsenal de Machine Learning. Con práctica y experimentación, puedes aplicarlas a problemas reales de predicción temporal.